# Week 7 Exercise — "The Bug Count Is Right" 🐛🔢

> **Disclaimer:** Please ensure you have the `buggy_dataset_nl.jsonl` file placed in *this same directory*. You can download it from this [Google Drive folder](https://drive.google.com/drive/folders/1AaLC3nnm6gJLU66R7mAYFuSQRkxC0WqV?usp=drive_link).

**Description**: Inspired by Week 7's "The Price Is Right" capstone (where we QLoRA fine-tune Llama 3.2 to predict product prices), this exercise applies the same paradigm to the **Bug Pipeline** from Weeks 1–6: **predicting the number of bugs** in a Python code snippet.

We demonstrate the full Week 7 workflow:
1. **Data Preparation** — Load, explore, and split the dataset; build prompt pairs.
2. **Simple Neural Network** — HashingVectorizer → PyTorch MLP as a lightweight baseline.
3. **LoRA Fine-Tuning** — Freeze a pre-trained CodeT5-small and train **only LoRA adapter modules** (the core Week 7 skill).
4. **Ollama Llama 3.2** — Zero-shot and few-shot baselines via local Ollama.
5. **Gradio UI & Results** — Interactive comparison and final evaluation charts.

The key learning: **freeze the base model, train only small adapter layers** — same as QLoRA on Llama 3.2, but sized to run locally.

**Dependencies:** If you see `No module named pip`, install from the repo root: `uv sync` or `uv pip install ...`. Then restart the kernel.

In [ ]:
%pip install -q torch scikit-learn matplotlib seaborn pandas gradio requests transformers peft accelerate>=0.26.0

In [ ]:
import json
import pandas as pd
import numpy as np
import random
import re
import matplotlib.pyplot as plt
import seaborn as sns
import requests
from time import time

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score

# Reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Phase 1: Data Preparation & Visualization
Load the 60 items from `buggy_dataset_nl.jsonl`, explore the distribution, and create train/test splits.

In [ ]:
DATA_PATH = "buggy_dataset_nl.jsonl"

with open(DATA_PATH, "r") as f:
    dataset = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(dataset)
print(f"Loaded {len(df)} samples")
print(f"\nnum_bugs distribution:\n{df['num_bugs'].value_counts().sort_index()}")
print(f"\nlevel distribution:\n{df['level'].value_counts()}")
df[['id', 'level', 'description', 'num_bugs', 'bug_types']].head(5)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df['num_bugs'].hist(bins=[0.5, 1.5, 2.5, 3.5], ax=axes[0], color='coral', edgecolor='black', rwidth=0.7)
axes[0].set_title('Distribution of num_bugs')
axes[0].set_xlabel('num_bugs'); axes[0].set_ylabel('Count'); axes[0].set_xticks([1, 2, 3])

df['level'].value_counts().reindex(['easy','medium','hard']).plot.bar(
    ax=axes[1], color=['#4CAF50','#FFC107','#F44336'], edgecolor='black')
axes[1].set_title('Difficulty Levels'); axes[1].tick_params(axis='x', rotation=0)

df.groupby('level')['num_bugs'].mean().reindex(['easy','medium','hard']).plot.bar(
    ax=axes[2], color=['#4CAF50','#FFC107','#F44336'], edgecolor='black')
axes[2].set_title('Avg num_bugs by Level'); axes[2].tick_params(axis='x', rotation=0)

plt.tight_layout(); plt.show()

In [ ]:
# Train/Test split — 50 train, 10 test
train_df, test_df = train_test_split(df, test_size=10, random_state=42, stratify=df['num_bugs'])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
print(f"Train: {len(train_df)} | Test: {len(test_df)}")
print(f"Train num_bugs: {dict(train_df['num_bugs'].value_counts().sort_index())}")
print(f"Test  num_bugs: {dict(test_df['num_bugs'].value_counts().sort_index())}")

---
## Phase 2: Simple Neural Network — Bug Count Regressor
A lightweight PyTorch baseline: HashingVectorizer (character n-grams) → 3-layer MLP → predicts `num_bugs`.

In [ ]:
# Vectorize
N_FEATURES = 512
vectorizer = HashingVectorizer(n_features=N_FEATURES, alternate_sign=False, analyzer='char_wb', ngram_range=(2, 4))

X_train_nn = vectorizer.fit_transform(train_df['buggy_code'].tolist()).toarray()
X_test_nn  = vectorizer.transform(test_df['buggy_code'].tolist()).toarray()
y_train_nn = train_df['num_bugs'].values.astype(np.float32)
y_test_nn  = test_df['num_bugs'].values.astype(np.float32)

class BugCountNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

model_nn = BugCountNN(N_FEATURES).to(device)
print(f"Simple NN parameters: {sum(p.numel() for p in model_nn.parameters()):,}")

In [ ]:
# Train
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_nn, dtype=torch.float32),
                  torch.tensor(y_train_nn, dtype=torch.float32)),
    batch_size=16, shuffle=True)

EPOCHS = 200
criterion = nn.MSELoss()
optimizer = optim.Adam(model_nn.parameters(), lr=0.001)

losses = []
for epoch in range(EPOCHS):
    model_nn.train()
    epoch_loss = 0.0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model_nn(Xb), yb)
        loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} — Loss: {losses[-1]:.4f}")

plt.figure(figsize=(10, 3))
plt.plot(losses, color='coral'); plt.title('Simple NN Training Loss'); plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Evaluate Simple NN
model_nn.eval()
with torch.no_grad():
    raw = model_nn(torch.tensor(X_test_nn, dtype=torch.float32).to(device)).cpu().numpy()
nn_preds = np.clip(np.round(raw), 1, 3).astype(int)
nn_mae = mean_absolute_error(y_test_nn, nn_preds)
nn_acc = accuracy_score(y_test_nn.astype(int), nn_preds)

print("=== Simple NN ===")
print(f"MAE: {nn_mae:.3f}  |  Accuracy: {nn_acc:.1%}")
print(f"Preds:  {nn_preds.tolist()}")
print(f"Actual: {y_test_nn.astype(int).tolist()}")

---
## Phase 3: LoRA Fine-Tuning — CodeT5-small 🔥

This is the **core Week 7 skill**: freeze a pre-trained model and **train only small LoRA adapter layers**.

We use [CodeT5-small](https://huggingface.co/Salesforce/codet5-small) (60M params) with **PEFT LoRA** — the same paradigm as QLoRA on Llama 3.2 from the course, but sized to run on CPU.

**What happens under the hood:**
- The entire CodeT5-small base model is **frozen** (no gradient updates)
- Small rank-decomposition matrices (**LoRA adapters**) are injected into selected attention modules
- Only these adapters (~0.3% of total params) get trained
- At inference, adapter weights are merged with the base for zero overhead

**Task:** Given buggy Python code as input, generate the number of bugs as output text (e.g., `"2"`).

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset

BASE_MODEL = "Salesforce/codet5-small"  # 60M params, runs on CPU

tokenizer_t5 = AutoTokenizer.from_pretrained(BASE_MODEL)
base_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

# Count base model params
total_params = sum(p.numel() for p in base_model.parameters())
print(f"Base model: {BASE_MODEL}")
print(f"Total parameters: {total_params:,}")

In [ ]:
# Apply LoRA — freeze base, train ONLY adapter modules
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,                      # Rank of the decomposition
    lora_alpha=32,            # Scaling factor
    lora_dropout=0.1,
    target_modules=["q", "v"],  # Only inject into attention Q and V projections
)

lora_model = get_peft_model(base_model, lora_config)

# Show the difference: trainable vs frozen
lora_model.print_trainable_parameters()

In [ ]:
# Prepare dataset for seq2seq training
PROMPT_TEMPLATE = "Count bugs in this Python code: {code}"
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 8

class BugCountDataset(Dataset):
    def __init__(self, dataframe, tokenizer):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        input_text = PROMPT_TEMPLATE.format(code=row['buggy_code'])
        target_text = str(row['num_bugs'])
        
        input_enc = self.tokenizer(
            input_text, max_length=MAX_INPUT_LEN, truncation=True, padding='max_length', return_tensors='pt'
        )
        target_enc = self.tokenizer(
            target_text, max_length=MAX_TARGET_LEN, truncation=True, padding='max_length', return_tensors='pt'
        )
        
        labels = target_enc['input_ids'].squeeze()
        labels[labels == self.tokenizer.pad_token_id] = -100  # Ignore padding in loss
        
        return {
            'input_ids': input_enc['input_ids'].squeeze(),
            'attention_mask': input_enc['attention_mask'].squeeze(),
            'labels': labels
        }

train_dataset = BugCountDataset(train_df, tokenizer_t5)
test_dataset  = BugCountDataset(test_df, tokenizer_t5)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples:  {len(test_dataset)}")

# Preview one example
sample = train_dataset[0]
print(f"\nSample input tokens: {sample['input_ids'].shape}")
print(f"Sample decoded input: {tokenizer_t5.decode(sample['input_ids'], skip_special_tokens=True)[:200]}...")

In [ ]:
# Train with HuggingFace Seq2SeqTrainer
training_args = Seq2SeqTrainingArguments(
    output_dir="./lora_bug_count",
    num_train_epochs=30,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=3e-4,
    weight_decay=0.01,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    report_to="none",
    fp16=False,  # CPU-safe
)

trainer = Seq2SeqTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer_t5,
)

print("Starting LoRA fine-tuning (this may take a few minutes on CPU)...")
t0 = time()
trainer.train()
print(f"\nTraining completed in {time() - t0:.0f}s")

In [ ]:
# Plot LoRA training loss
log_history = trainer.state.log_history
train_losses = [h['loss'] for h in log_history if 'loss' in h]
eval_losses  = [h['eval_loss'] for h in log_history if 'eval_loss' in h]

fig, ax = plt.subplots(figsize=(10, 4))
if train_losses:
    ax.plot(train_losses, label='Train Loss', color='#FF6B6B', linewidth=1.5)
if eval_losses:
    eval_x = np.linspace(0, len(train_losses)-1, len(eval_losses))
    ax.plot(eval_x, eval_losses, label='Eval Loss', color='#4ECDC4', linewidth=1.5, linestyle='--')
ax.set_title('LoRA Fine-Tuning — Training & Eval Loss', fontsize=14, fontweight='bold')
ax.set_xlabel('Logging Step'); ax.set_ylabel('Loss'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Evaluate LoRA model on test set
lora_model.eval()
lora_preds = []

for i in range(len(test_df)):
    code = test_df.iloc[i]['buggy_code']
    input_text = PROMPT_TEMPLATE.format(code=code)
    inputs = tokenizer_t5(input_text, max_length=MAX_INPUT_LEN, truncation=True, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output_ids = lora_model.generate(**inputs, max_new_tokens=MAX_TARGET_LEN)
    
    decoded = tokenizer_t5.decode(output_ids[0], skip_special_tokens=True).strip()
    # Parse integer from output
    nums = re.findall(r'\d+', decoded)
    pred = int(nums[0]) if nums else 2  # default fallback
    pred = max(1, min(pred, 5))  # clamp
    lora_preds.append(pred)

lora_preds = np.array(lora_preds)
y_test_actual = test_df['num_bugs'].values

lora_mae = mean_absolute_error(y_test_actual, lora_preds)
lora_acc = accuracy_score(y_test_actual, lora_preds)

print("=== LoRA Fine-Tuned CodeT5-small ===")
print(f"MAE: {lora_mae:.3f}  |  Accuracy: {lora_acc:.1%}")
print(f"Preds:  {lora_preds.tolist()}")
print(f"Actual: {y_test_actual.tolist()}")

---
## Phase 4: Ollama Llama 3.2 — Zero-Shot & Few-Shot
Compare the fine-tuned model against a general-purpose LLM using the local Ollama API.

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2"

def ask_ollama(prompt, model=OLLAMA_MODEL):
    try:
        resp = requests.post(OLLAMA_URL, json={
            "model": model, "prompt": prompt, "stream": False,
            "options": {"temperature": 0.0}
        }, timeout=120)
        resp.raise_for_status()
        return resp.json().get("response", "").strip()
    except Exception as e:
        return f"ERROR: {e}"

def parse_bug_count(text):
    nums = re.findall(r'\b([1-9]\d{0,1})\b', text)
    return min(int(nums[0]), 10) if nums else -1

# Test connection
print(f"Ollama test: '{ask_ollama('Say hello in one word.')}'")

In [ ]:
# Build prompts
ZS_TEMPLATE = """You are a Python bug detection expert. Count the distinct bugs in this code.
Reply with ONLY a single integer.

```python
{code}
```

Number of bugs:"""

# Few-shot: one example each of 1, 2, 3 bugs from training set
FS_PREFIX = "You are a Python bug detection expert. Count the distinct bugs in code.\n\nExamples:\n\n"
for n in [1, 2, 3]:
    s = train_df[train_df['num_bugs'] == n].iloc[0]
    FS_PREFIX += f"```python\n{s['buggy_code']}\n```\nNumber of bugs: {s['num_bugs']}\n\n"

FS_TEMPLATE = FS_PREFIX + """Now count bugs in this code. Reply with ONLY a single integer.

```python
{code}
```

Number of bugs:"""

print(f"Zero-shot template: {len(ZS_TEMPLATE)} chars")
print(f"Few-shot template:  {len(FS_TEMPLATE)} chars")

In [ ]:
# Run Ollama on test set
zs_preds, fs_preds = [], []
test_actual = test_df['num_bugs'].tolist()

print("Running Ollama predictions...\n")
for i, code in enumerate(test_df['buggy_code']):
    print(f"  [{i+1}/{len(test_df)}]", end=" ")
    t0 = time()
    
    zs = parse_bug_count(ask_ollama(ZS_TEMPLATE.format(code=code)))
    fs = parse_bug_count(ask_ollama(FS_TEMPLATE.format(code=code)))
    zs_preds.append(zs); fs_preds.append(fs)
    
    print(f"ZS={zs} FS={fs} Actual={test_actual[i]} ({time()-t0:.1f}s)")

print("\nDone!")

In [ ]:
# Evaluate Ollama
def evaluate(actual, predicted, label):
    a, p = np.array(actual), np.array(predicted)
    valid = p > 0
    if valid.sum() == 0:
        print(f"{label}: No valid predictions (is Ollama running?)")
        return 999.0, 0.0
    mae = mean_absolute_error(a[valid], p[valid])
    acc = accuracy_score(a[valid], p[valid])
    print(f"{label}: MAE={mae:.3f} | Accuracy={acc:.1%} | Valid={valid.sum()}/{len(p)}")
    return mae, acc

zs_mae, zs_acc = evaluate(test_actual, zs_preds, "Zero-Shot")
fs_mae, fs_acc = evaluate(test_actual, fs_preds, "Few-Shot")

---
## Phase 5: Gradio Comparison UI & Final Results

In [ ]:
import gradio as gr

SAMPLE_CODE = test_df.iloc[0]['buggy_code']

def predict_simple_nn(code):
    if not code.strip(): return "Paste some Python code."
    feats = vectorizer.transform([code]).toarray()
    model_nn.eval()
    with torch.no_grad():
        raw = model_nn(torch.tensor(feats, dtype=torch.float32).to(device)).cpu().item()
    pred = int(np.clip(round(raw), 1, 3))
    return f"🔢 **{pred}** bugs detected\n\n_(raw: {raw:.3f})_"

def predict_lora(code):
    if not code.strip(): return "Paste some Python code."
    inp = tokenizer_t5(PROMPT_TEMPLATE.format(code=code), max_length=MAX_INPUT_LEN, truncation=True, return_tensors='pt').to(device)
    lora_model.eval()
    with torch.no_grad():
        out = lora_model.generate(**inp, max_new_tokens=MAX_TARGET_LEN)
    decoded = tokenizer_t5.decode(out[0], skip_special_tokens=True).strip()
    nums = re.findall(r'\d+', decoded)
    pred = max(1, min(int(nums[0]), 5)) if nums else 2
    return f"🔢 **{pred}** bugs detected\n\n_(model output: '{decoded}')_"

def predict_ollama(code, mode):
    if not code.strip(): return "Paste some Python code."
    tmpl = ZS_TEMPLATE if mode == "Zero-Shot" else FS_TEMPLATE
    resp = ask_ollama(tmpl.format(code=code))
    count = parse_bug_count(resp)
    return f"🔢 **{count}** bugs detected\n\n_Response: {resp[:300]}_"

def results_chart():
    labels = ['Simple NN', 'LoRA\nCodeT5', 'Llama 3.2\nZero-Shot', 'Llama 3.2\nFew-Shot']
    maes = [nn_mae, lora_mae, zs_mae if zs_mae < 100 else 0, fs_mae if fs_mae < 100 else 0]
    accs = [nn_acc, lora_acc, zs_acc, fs_acc]
    colors = ['#FF6B6B', '#9B59B6', '#4ECDC4', '#45B7D1']
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, vals, title, ylabel in [
        (ax1, maes, 'MAE (lower = better)', 'MAE'),
        (ax2, accs, 'Accuracy (higher = better)', 'Accuracy')
    ]:
        bars = ax.bar(labels, vals, color=colors, edgecolor='black', linewidth=0.8)
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel(ylabel); ax.grid(axis='y', alpha=0.3)
        for b, v in zip(bars, vals):
            fmt = f'{v:.2f}' if ylabel == 'MAE' else f'{v:.0%}'
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, fmt, ha='center', fontweight='bold')
    if ylabel == 'Accuracy': ax2.set_ylim(0, 1.15)
    plt.suptitle('🐛🔢 The Bug Count Is Right — Results', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig('results_chart.png', dpi=150, bbox_inches='tight'); plt.close(fig)
    return 'results_chart.png'

with gr.Blocks(title="The Bug Count Is Right 🐛🔢", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🐛🔢 The Bug Count Is Right\nPaste buggy Python code → predict the number of bugs.")
    with gr.Tabs():
        with gr.TabItem("🧠 Simple NN"):
            with gr.Row():
                c1 = gr.Code(label="Code", language="python", value=SAMPLE_CODE)
                o1 = gr.Markdown()
            gr.Button("Predict", variant="primary").click(predict_simple_nn, c1, o1)
        
        with gr.TabItem("🔥 LoRA CodeT5"):
            gr.Markdown("*Fine-tuned with LoRA adapters — only ~0.3% of params trained*")
            with gr.Row():
                c2 = gr.Code(label="Code", language="python", value=SAMPLE_CODE)
                o2 = gr.Markdown()
            gr.Button("Predict", variant="primary").click(predict_lora, c2, o2)
        
        with gr.TabItem("🦙 Ollama Llama 3.2"):
            with gr.Row():
                c3 = gr.Code(label="Code", language="python", value=SAMPLE_CODE)
                with gr.Column():
                    mode = gr.Radio(["Zero-Shot", "Few-Shot"], value="Zero-Shot", label="Mode")
                    o3 = gr.Markdown()
            gr.Button("Predict", variant="primary").click(predict_ollama, [c3, mode], o3)
        
        with gr.TabItem("📊 Results"):
            chart_img = gr.Image()
            gr.Button("Show Chart", variant="primary").click(results_chart, None, chart_img)

demo.launch()

In [ ]:
# Final standalone results chart
labels = ['Simple NN', 'LoRA CodeT5', 'Llama 3.2\nZero-Shot', 'Llama 3.2\nFew-Shot']
maes = [nn_mae, lora_mae, zs_mae if zs_mae < 100 else 0, fs_mae if fs_mae < 100 else 0]
accs = [nn_acc, lora_acc, zs_acc, fs_acc]
colors = ['#FF6B6B', '#9B59B6', '#4ECDC4', '#45B7D1']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, vals, title, yl in [(ax1, maes, 'MAE (lower=better)', 'MAE'), (ax2, accs, 'Accuracy (higher=better)', 'Accuracy')]:
    bars = ax.bar(labels, vals, color=colors, edgecolor='black')
    ax.set_title(title, fontweight='bold', fontsize=13); ax.set_ylabel(yl); ax.grid(axis='y', alpha=0.3)
    for b, v in zip(bars, vals):
        fmt = f'{v:.2f}' if yl == 'MAE' else f'{v:.0%}'
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02, fmt, ha='center', fontweight='bold', fontsize=11)
ax2.set_ylim(0, 1.15)
plt.suptitle('🐛🔢 "The Bug Count Is Right" — Final Results', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout(); plt.show()